In [11]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load and Pre-process Data[cite: 66]
df = pd.read_csv('uber.csv').dropna()
df['hour'] = pd.to_datetime(df['pickup_datetime']).dt.hour
df = df.drop(columns=['key', 'pickup_datetime'], errors='ignore')

# Calculate Haversine distance[cite: 72]
lat1, lon1, lat2, lon2 = map(np.radians, [df['pickup_latitude'], df['pickup_longitude'], df['dropoff_latitude'], df['dropoff_longitude']])
df['distance'] = 6371 * (2 * np.arcsin(np.sqrt(np.sin((lat2 - lat1)/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1)/2.0)**2)))

# 2. Remove Outliers using IQR[cite: 70]
for col in ['fare_amount', 'distance']:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    df = df[(df[col] >= q1 - 1.5*(q3-q1)) & (df[col] <= q3 + 1.5*(q3-q1))]

# 3. Check Correlation[cite: 73]
print("Correlation with Fare:\n", df.corr()['fare_amount'])

# 4. Train Models & Evaluate (Linear Regression & Random Forest)[cite: 66]
X = df[['passenger_count', 'hour', 'distance', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']]
y = df['fare_amount']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for name, model in [("Linear Regression", LinearRegression()), ("Random Forest", RandomForestRegressor(n_estimators=50, random_state=42))]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"\n{name} -> RMSE: {np.sqrt(mean_squared_error(y_test, pred)):.4f} | R2: {r2_score(y_test, pred):.4f}")

Correlation with Fare:
 Unnamed: 0          -0.001236
fare_amount          1.000000
pickup_longitude     0.014309
pickup_latitude     -0.014140
dropoff_longitude    0.014412
dropoff_latitude    -0.014282
passenger_count      0.016415
hour                 0.006141
distance             0.754566
Name: fare_amount, dtype: float64

Linear Regression -> RMSE: 2.3637 | R2: 0.5966

Random Forest -> RMSE: 2.1812 | R2: 0.6565
